# agent-core：真实模型与通用循环实验

独立运行本 Notebook，学习 Agent 状态、工具协议、事件顺序和 follow-up。只使用 fox_ai 与 fox_agent_core；文件工具、Session、Skill 和压缩在 [coding-agent 实验](../fox_coding_agent/CODING_AGENT_LAB.ipynb)。

配套 [架构指南](ARCHITECTURE_GUIDE.md)。选择 Python 3.14+ 内核，先 `uv sync`，按顺序运行。真实模型请求可能收费，密钥从环境变量或项目 `.env` 读取。本次拆分已清空旧输出，避免把旧结果当作新实验的验证结果。


## 0. 环境与模型配置

fox_ai 统一模型协议，Agent 管理状态，agent_loop 推进工具闭环。此实验不导入 coding-agent。


In [ ]:
import sys
import importlib.util
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("请选择项目 Python 3.14+ 内核，然后重新从第一格运行。")
required = ["openai", "anthropic", "pydantic", "jsonschema", "yaml", "IPython"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"当前内核缺少 {missing}；请在项目根目录运行 uv sync 并切换到 .venv 内核。")

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "packages/fox_agent_core/src/agent.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("请从 FoxCode 仓库内打开 Notebook。")
if str(REPO_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "packages"))

import asyncio
import getpass
import html
import json
import os
import re
import tempfile
import time
from collections import Counter
from contextvars import ContextVar
from dataclasses import asdict
from IPython.display import HTML, Markdown, display

from fox_ai.src import (
    Model, AssistantMessage, UserMessage, ToolCall, ToolResultMessage,
    TextContent, stream_simple,
)
from fox_agent_core.src import Agent, AgentOptions, AgentToolResult

print("Python:", sys.version.split()[0])
print("Agent 源码:", REPO_ROOT / "packages/fox_agent_core")

### 配置中哪些数字代表真实服务，哪些是教学预算？

- `MODEL_ID` / `BASE_URL` 是真实请求目的地，默认与已有 DeepSeek 实验一致。
- `LAB_CONTEXT_WINDOW` 是本实验交给模型适配层 的**本地上下文预算**，不宣称它等于服务商实际最大窗口。
- `OUTPUT_TOKENS` 控制单次回答的输出上限；`MAX_REAL_REQUESTS` 限制本次 Notebook 的模型请求总数，包含摘要调用。
- 本实验关闭 DeepSeek thinking，以便集中观察 Agent 架构；改成其他兼容服务时，应相应调整 `sampling_params`。

读取 `.env` 只匹配指定的一个密钥名，不打印文件内容，也不把整份环境配置传给模型。找不到时才出现隐藏输入框。

In [ ]:
MODEL_ID = os.getenv("FOX_AGENT_MODEL_ID", "deepseek-flash")
BASE_URL = os.getenv("FOX_AGENT_BASE_URL", "https://api.deepseek.com")
API_KEY_ENV = "DEEPSEEK_API_KEY"
LAB_CONTEXT_WINDOW = 32768       # 本地教学预算，不是对服务商规格的声明
OUTPUT_TOKENS = 1200
MAX_REAL_REQUESTS = 12

def load_key(name):
    value = os.environ.get(name, "").strip()
    env_file = REPO_ROOT / ".env"
    if not value and env_file.is_file():
        for line in env_file.read_text(encoding="utf-8-sig").splitlines():
            left, sep, right = line.strip().removeprefix("export ").partition("=")
            if sep and left.strip() == name:
                value = right.strip()
                if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
                    value = value[1:-1]
                else:
                    value = value.split(" #", 1)[0].strip()
                break
    return value or getpass.getpass(f"请输入 {name}（不回显）：").strip()

API_KEY = load_key(API_KEY_ENV)
if not API_KEY:
    raise RuntimeError("没有可用密钥。请设置 DEEPSEEK_API_KEY 后重跑此单元。")

model = Model(
    id=MODEL_ID, name="真实模型 · Agent 学习实验", provider="deepseek",
    api="openai-completions", base_url=BASE_URL, input=["text"],
    reasoning=False, context_window=LAB_CONTEXT_WINDOW, max_tokens=OUTPUT_TOKENS,
    compat={"supportsStrictMode": False},
)
STREAM_OPTIONS = {
    "api_key": API_KEY, "max_tokens": OUTPUT_TOKENS,
    "timeout_ms": 90000, "max_retries": 0,
    "sampling_params": {"extra_body": {"thinking": {"type": "disabled"}}},
}
print({"model": model.id, "api": model.api, "base_url": model.base_url,
       "local_context_budget": model.context_window, "key_loaded": bool(API_KEY)})

### 一组观察辅助函数（可以折叠）

下面的代码只负责记录请求和渲染事件。`observed_stream` **始终调用真实的 `stream_simple`**；包装器只记录用量，不会改写模型回答。

观察表记录三类信息：模型请求及 token 用量、Agent 生命周期事件、完成消息。不要打印 `STREAM_OPTIONS`，因为其中包含密钥。

In [ ]:
PHASE = ContextVar("fox_agent_lab_phase", default="未标记")
REQUESTS = []
EVENTS = []
ATTACHED = {}

def safe_text(value):
    text = str(value).replace(API_KEY, "[REDACTED]")
    return re.sub(r"sk-[A-Za-z0-9_-]{12,}", "[REDACTED]", text)

def table(rows):
    if not rows:
        print("（没有记录）")
        return
    columns = list(rows[0])
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(safe_text(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

class MeasuredStream:
    def __init__(self, inner, row):
        self.inner, self.row = inner, row

    def __aiter__(self):
        return self.inner.__aiter__()

    async def result(self):
        message = await self.inner.result()
        usage = message.usage
        self.row.update({
            "status": message.stop_reason,
            "seconds": round(time.perf_counter() - self.row["started"], 2),
            "input": usage.input, "output": usage.output,
            "cache_read": usage.cache_read, "cache_write": usage.cache_write,
            "total": usage.total_tokens or (usage.input + usage.output + usage.cache_read + usage.cache_write),
        })
        return message

    async def aclose(self):
        await self.inner.aclose()
        if self.row["status"] == "pending":
            self.row["status"] = "closed"

def observed_stream(request_model, context, options):
    if len(REQUESTS) >= MAX_REAL_REQUESTS:
        raise RuntimeError("达到本次 Notebook 的真实请求上限；检查用量后再调整 MAX_REAL_REQUESTS。")
    row = {
        "number": len(REQUESTS) + 1, "phase": PHASE.get(),
        "kind": "agent",
        "model": request_model.id, "messages": len(context.messages),
        "tools": [t.name for t in context.tools or []],
        "started": time.perf_counter(), "status": "pending",
    }
    REQUESTS.append(row)
    try:
        return MeasuredStream(stream_simple(request_model, context, options), row)
    except Exception:
        row["status"] = "request_setup_error"
        raise

def message_text(message):
    if isinstance(message.content, str):
        return message.content
    return "\n".join(c.text for c in message.content if isinstance(c, TextContent))

def final_text(agent_or_harness):
    last = next((m for m in reversed(agent_or_harness.state.messages)
                 if isinstance(m, AssistantMessage)), None)
    return message_text(last) if last else ""

def attach(subject, label):
    old = ATTACHED.pop(id(subject), None)
    if old:
        old()

    def on_event(event, cancel_event):
        row = {"phase": PHASE.get(), "owner": label, "type": event.type}
        if event.type.startswith("tool_execution"):
            row.update({"tool": event.tool_name, "call_id": event.tool_call_id})
        if event.type == "tool_execution_start":
            row["args"] = event.args
            print(f"\n[工具开始] {event.tool_name} {safe_text(event.args)}")
        elif event.type == "tool_execution_end":
            row["is_error"] = event.is_error
            print(f"[工具结束] {event.tool_name} / error={event.is_error}")
        elif event.type == "message_update":
            delta = event.assistant_message_event
            if delta.type == "text_delta":
                print(safe_text(delta.delta), end="", flush=True)
        elif event.type in ("turn_start", "agent_end", "compaction_start", "compaction_end", "compaction_error"):
            print(f"\n[{event.type}]")
        EVENTS.append(row)

    ATTACHED[id(subject)] = subject.subscribe(on_event)

async def run_step(subject, text, phase):
    token = PHASE.set(phase)
    start = len(REQUESTS)
    try:
        operation = subject.prompt(text)
        await asyncio.wait_for(operation, timeout=240)
        if subject.state.error_message:
            raise RuntimeError(safe_text(subject.state.error_message))
        print(f"\n本步骤新增真实请求：{len(REQUESTS) - start}")
        return final_text(subject)
    except Exception as exc:
        raise RuntimeError(safe_text(exc)) from None
    finally:
        PHASE.reset(token)

def show_messages(subject, limit=14):
    messages = subject.state.messages
    rows = []
    for index, message in enumerate(messages[-limit:], max(0, len(messages) - limit)):
        calls = [c.name for c in message.content if isinstance(c, ToolCall)] if isinstance(message.content, list) else []
        rows.append({"index": index, "role": message.role,
                     "calls": ", ".join(calls), "call_id": getattr(message, "tool_call_id", ""),
                     "preview": message_text(message)[:120]})
    table(rows)

def assert_paired(messages):
    waiting = set()
    for message in messages:
        if isinstance(message, AssistantMessage):
            assert not waiting, f"上一轮仍有未配对工具调用：{waiting}"
            waiting = {c.id for c in message.content if isinstance(c, ToolCall)}
        elif isinstance(message, ToolResultMessage):
            assert message.tool_call_id in waiting, "出现没有对应调用的工具结果"
            waiting.remove(message.tool_call_id)
        elif isinstance(message, UserMessage):
            assert not waiting, "用户消息插入到了未完成的工具调用组中"
    assert not waiting, f"存在未完成工具调用：{waiting}"

## 1. 一次真实请求：Agent 层先提供什么？

此处只使用 `Agent`，没有 Session、Skill 和压缩。先观察 `agent_start → turn_start → message_* → turn_end → agent_end`。

**运行前预测**：没有工具、没有排队消息时，会请求模型几次？回答保存到哪里？

In [ ]:
basic = Agent(AgentOptions(
    initial_state={"model": model, "system_prompt": "请用简洁中文回答。", "tools": []},
    stream_fn=observed_stream, stream_options=STREAM_OPTIONS, max_turns=3,
))
attach(basic, "basic")
before = len(REQUESTS)
await run_step(basic, "用一句话说明：为什么工具执行后还需要把结果交给模型？", "01-普通响应")
assert len(REQUESTS) == before + 1
assert not basic.is_running
show_messages(basic)

**观察与面试复盘**：模型请求结束不需要再次轮询，因为没有新工作。Agent 仍保存了用户和助手两条消息，说明 Agent 是有状态对象；它尚未把历史写到磁盘。

## 2. 自定义工具：模型提出调用，Python 负责执行

我们提供一个 `capacity_budget` 工具，计算 `workers × 每个 worker 的并发上限`。这里的计算只是教学例子，不是生产容量评估公式。

关键不是乘法，而是三段边界：**Schema 声明 → 运行时校验和本地执行 → ToolResult 回传模型**。不在提示词中预先提供工具结果。

In [ ]:
EXECUTED_BUDGETS = []

class CapacityBudgetTool:
    name = "capacity_budget"
    label = "并发额度"
    description = "计算本练习的总并发额度；必须用此工具获取结果。"
    parameters = {"type": "object", "properties": {
        "workers": {"type": "integer", "minimum": 1},
        "concurrency_per_worker": {"type": "integer", "minimum": 1},
    }, "required": ["workers", "concurrency_per_worker"], "additionalProperties": False}

    async def execute(self, call_id, params, cancel_event=None, on_update=None):
        result = params["workers"] * params["concurrency_per_worker"]
        EXECUTED_BUDGETS.append({"call_id": call_id, **params, "result": result})
        return AgentToolResult(
            content=[TextContent(text=json.dumps({"concurrency_budget": result}))],
            details={"formula": "workers * concurrency_per_worker"},
        )

budget_tool = CapacityBudgetTool()
tool_agent = Agent(AgentOptions(
    initial_state={"model": model, "tools": [budget_tool],
                   "system_prompt": "你正在演示工具闭环。计算题必须调用 capacity_budget，不能自行给出结果。"},
    stream_fn=observed_stream, stream_options=STREAM_OPTIONS, max_turns=4,
))
attach(tool_agent, "custom-tool")
await run_step(tool_agent, "请调用 capacity_budget：workers=4，每个 worker 的并发上限为 37，然后解释工具给出的结果。", "02-自定义工具")
assert EXECUTED_BUDGETS, "模型没有调用工具，请检查端点是否支持 tool calling，并重跑本实验。"
assert EXECUTED_BUDGETS[-1]["result"] == 148
assert_paired(tool_agent.state.messages)
table(EXECUTED_BUDGETS)
show_messages(tool_agent)

**应看到的证据**：`EXECUTED_BUDGETS` 有本地执行记录；消息顺序包含 assistant 工具调用和 toolResult；随后才是模型解释。一次 `prompt` 因此包含多个 turn。

**面试追问**：如果模型传入字符串而非整数，应该在哪一层拒绝？如果工具名称相同但调用两次，怎样区分结果？答案分别是执行前的 Schema 校验，以及 `tool_call_id`。

## 3. follow-up：没有工具调用也可以继续

先排入一条 follow-up，再启动初始请求。循环处理完当前工作后消费 follow-up，因此一次 prompt 会包含至少两次真实请求。它是同一 Agent 的队列，不是 Session，也不是新 Agent。


In [ ]:
queued = Agent(AgentOptions(
    initial_state={"model": model, "system_prompt": "请用简洁中文回答。", "tools": []},
    stream_fn=observed_stream, stream_options=STREAM_OPTIONS, max_turns=4,
))
attach(queued, "follow-up")
queued.follow_up(UserMessage(content="再用一句话解释：工具执行为什么不由模型服务直接完成？"))
request_start = len(REQUESTS)
await run_step(queued, "用一句话说明 Agent 的作用。", "03-follow-up")
assert len(REQUESTS) - request_start >= 2
assert len([m for m in queued.state.messages if isinstance(m, UserMessage)]) == 2
show_messages(queued)


## 4. 复盘与用量

用事件和消息解释：一次 prompt 与一次 turn 有什么区别？为什么工具调用需要 ID？follow-up 在哪个时机消费？Agent 保存了哪些内存状态，哪些信息仍没有持久化？

本实验不保存 Session。继续学习文件历史与压缩，请打开另一本 coding-agent Notebook。


In [ ]:
table([{k: r.get(k, "") for k in ["number", "phase", "messages", "status", "seconds", "total"]} for r in REQUESTS])
print("真实请求次数:", len(REQUESTS))
print("provider 报告的累计 token:", sum(r.get("total", 0) for r in REQUESTS))
for unsubscribe in ATTACHED.values():
    unsubscribe()
ATTACHED.clear()
